[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/aipi590-challenge-3/blob/main/notebooks/challenge3.ipynb)

# AIPI 590 — Challenge 3: RL in the Physical World

Train a manipulation policy in MuJoCo simulation (FetchReach-v4) using SAC + Hindsight Experience Replay, then analyze the sim-to-real gap against a real 6-DOF arm platform.

## 0. Install

Run this cell first. If Colab prompts a runtime restart, restart and continue from the next cell — do not re-run this one.

In [1]:
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'mujoco>=3.0.0',
    'gymnasium-robotics>=1.2.0',
    'stable-baselines3[extra]>=2.3.0',
    'moviepy>=1.0.3',
    'imageio[ffmpeg]',
], check=True)

CompletedProcess(args=['pip', 'install', '-q', 'mujoco>=3.0.0', 'gymnasium-robotics>=1.2.0', 'stable-baselines3[extra]>=2.3.0', 'moviepy>=1.0.3', 'imageio[ffmpeg]'], returncode=0)

## 1. Setup

In [2]:
import sys, urllib.request
from pathlib import Path

utils_path = Path('/content/colab_utils.py')
if not utils_path.exists():
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/jonasneves/aipi590-challenge-3/main/scripts/colab_utils.py',
        str(utils_path),
    )

sys.path.insert(0, '/content')
import colab_utils

REPO = colab_utils.prepare_notebook(pull_latest=True)
publish_artifacts = colab_utils.publish_artifacts
save_notebook = colab_utils.save_notebook
print(f'Repo ready at {REPO}')

Repo ready at /content/aipi590-challenge-3


In [3]:
import gymnasium as gym
import gymnasium_robotics
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import base64
from IPython.display import HTML, display

from stable_baselines3 import SAC
from stable_baselines3.her.her_replay_buffer import HerReplayBuffer
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor

gym.register_envs(gymnasium_robotics)

RESULTS   = REPO / 'results'
MODELS_DIR = RESULTS / 'models'
VIDEOS_DIR = RESULTS / 'videos'
PLOTS_DIR  = RESULTS / 'plots'
for d in [MODELS_DIR, VIDEOS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Imports OK')

Imports OK


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 2. Simulation Environment

**FetchReach-v4** places a 7-DOF Fetch robot arm in a MuJoCo scene and asks it to move the end-effector to a randomly sampled 3D target position. The reward is sparse: `0` on success (distance < 5 cm), `-1` otherwise. This makes vanilla RL nearly impossible without goal-conditioned hindsight — which is why HER is the right pairing.

In [4]:
env = gym.make('FetchReach-v4')
obs, _ = env.reset()

print('Observation keys:', list(obs.keys()))
print('  observation shape :', obs['observation'].shape)
print('  achieved_goal shape:', obs['achieved_goal'].shape)
print('  desired_goal shape :', obs['desired_goal'].shape)
print('Action space:', env.action_space)
env.close()

Observation keys: ['observation', 'achieved_goal', 'desired_goal']
  observation shape : (10,)
  achieved_goal shape: (3,)
  desired_goal shape : (3,)
Action space: Box(-1.0, 1.0, (4,), float32)


## 3. Training: SAC + HER

**Why SAC?** Off-policy, sample-efficient, handles continuous action spaces well. Entropy regularization prevents premature convergence — useful when the reward signal is almost always `-1` early in training.

**Why HER?** Hindsight Experience Replay relabels failed trajectories as successes toward the goal that was actually reached. It converts sparse-reward failures into dense learning signal without reward shaping. `goal_selection_strategy='future'` selects relabeling goals from states later in the same episode — empirically the strongest variant (Andrychowicz et al., 2017).

In [ ]:
LiveChartCallback = colab_utils.LiveChartCallback

In [ ]:
TOTAL_TIMESTEPS = 200_000  # ~10 min on A100, ~25 min on T4

train_env = Monitor(gym.make('FetchReach-v4'))
eval_env  = Monitor(gym.make('FetchReach-v4'))

model = SAC(
    'MultiInputPolicy',
    train_env,
    replay_buffer_class=HerReplayBuffer,
    replay_buffer_kwargs=dict(
        n_sampled_goal=4,
        goal_selection_strategy='future',
    ),
    verbose=0,  # silenced — live chart replaces terminal output
    buffer_size=int(1e6),
    learning_rate=1e-3,
    gamma=0.95,
    batch_size=256,
    policy_kwargs=dict(net_arch=[256, 256, 256]),
    tensorboard_log=str(RESULTS / 'tb'),
)

callbacks = [
    LiveChartCallback(update_freq=500, window=100),
    EvalCallback(
        eval_env,
        best_model_save_path=str(MODELS_DIR),
        log_path=str(RESULTS / 'eval_logs'),
        eval_freq=10_000,
        n_eval_episodes=20,
        verbose=0,
    ),
]

model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callbacks)
model.save(str(MODELS_DIR / 'sac_her_fetchreach_final'))
print('Training complete.')

In [6]:
eval_log = np.load(str(RESULTS / 'eval_logs' / 'evaluations.npz'))

timesteps    = eval_log['timesteps']
mean_reward  = eval_log['results'].mean(axis=1)
success_rate = (eval_log['results'] == 0).mean(axis=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(timesteps, mean_reward)
ax1.set_xlabel('Timesteps')
ax1.set_ylabel('Mean episode reward')
ax1.set_title('FetchReach-v4 — SAC+HER')
ax1.grid(alpha=0.3)

ax2.plot(timesteps, success_rate)
ax2.set_xlabel('Timesteps')
ax2.set_ylabel('Success rate')
ax2.set_ylim(0, 1)
ax2.set_title('FetchReach-v4 — Success Rate')
ax2.grid(alpha=0.3)

fig.tight_layout()
plot_path = PLOTS_DIR / 'training_curves.png'
fig.savefig(str(plot_path), dpi=150)
plt.show()
print(f'Saved to {plot_path}')

Saved to /content/aipi590-challenge-3/results/plots/training_curves.png


## 4. Evaluation

In [ ]:
import subprocess, os
subprocess.Popen(['Xvfb', ':1', '-screen', '0', '1024x768x24'])
os.environ['DISPLAY'] = ':1'

model = SAC.load(str(MODELS_DIR / 'best_model'), env=gym.make('FetchReach-v4'))

record_env = gym.make('FetchReach-v4', render_mode='rgb_array')
record_env = gym.wrappers.RecordVideo(
    record_env,
    str(VIDEOS_DIR),
    episode_trigger=lambda ep: True,
    name_prefix='fetchreach',
)

n_episodes = 5
successes = 0
for ep in range(n_episodes):
    obs, _ = record_env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = record_env.step(action)
        done = terminated or truncated
    if info.get('is_success', False):
        successes += 1

record_env.close()
print(f'Success rate: {successes}/{n_episodes}')

videos = sorted(Path(VIDEOS_DIR).glob('*.mp4'))
if videos:
    b64 = base64.b64encode(videos[-1].read_bytes()).decode()
    display(HTML(f'<video controls width="480"><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
else:
    print('No video found — check VIDEOS_DIR.')

## 5. Sim-to-Real Analysis

Training a policy in MuJoCo and deploying it on a physical arm exposes a structured set of gaps. The analysis below is grounded in the **reBot-DevArm** platform (6-DOF, Robostride/Damiao actuators, ROS 2 Humble), which represents a realistic target for deploying a policy trained in an environment like FetchReach-v4.

---

### 5.1 Actuator Fidelity

FetchReach-v4 uses **idealized velocity control**: the policy outputs a 4D delta in Cartesian space and MuJoCo solves inverse kinematics internally, applying exact joint velocities at every simulation step (~2 ms). Real actuators are substantially messier:

- **Gear backlash.** Robostride and Damiao motors use cycloidal or harmonic reducers with measurable dead-band. Commands below the dead-band threshold produce no motion; commands above it produce a step. The policy never encounters this in simulation.
- **Torque limits and thermal drift.** Real motors derate as they heat. A joint near its torque limit in simulation may saturate unpredictably on hardware after several minutes of operation.
- **Control loop latency.** The reBot-DevArm's ROS 2 control loop runs at ~100 Hz (~10 ms). MuJoCo steps are sub-millisecond. A policy trained at sim timestep resolution implicitly assumes zero actuator lag — a gap that compounds across a 7-step reaching trajectory.

**Mitigation:** randomize actuator delay, add gear backlash noise to actions, and cap joint velocity commands to real motor specs.

---

### 5.2 Contact and Collision Modeling

MuJoCo models contacts as convex-hull collisions with soft constraint forces. For pure reaching (no contact with the goal) this is largely irrelevant. It becomes the dominant failure mode the moment the task extends to **grasping**. Real gripper finger compliance, object surface friction variability, and micro-slip during grasp are not captured by MuJoCo's rigid-body contact model.

**Mitigation:** system identification of friction parameters and gripper compliance before extending sim-trained policies to manipulation.

---

### 5.3 Observation Noise

The FetchReach policy receives ground-truth joint positions, velocities, and goal position — all noiseless from the MuJoCo state vector. On the reBot-DevArm:

- Joint positions come from **magnetic encoders** (~0.1° resolution on Damiao series) with electrical noise at high speeds. Velocity is numerically differentiated, amplifying noise.
- The goal position is estimated from a **depth camera**, introducing 2–5 mm of spatial uncertainty and pipeline latency.

A policy trained on clean observations has learned to exploit their absence of noise rather than be robust to it.

**Mitigation:** add Gaussian noise to joint and goal observations during training.

---

### 5.4 Zero Calibration Drift

reBot-DevArm requires explicit zero-point calibration at each power cycle. Small per-joint errors (1–2°) compound through the kinematic chain: at 650 mm reach, a 2° wrist error produces ~22 mm of end-effector position error — well above the 50 mm success threshold. No domain randomization covers this; it requires a physical calibration procedure before deployment.

---

### 5.5 Domain Randomization Strategy

| Parameter | Range | Rationale |
|---|---|---|
| Action delay | 0–20 ms | ROS 2 control loop jitter |
| Joint obs noise | σ = 0.01 rad | Encoder resolution |
| Goal obs noise | σ = 3 mm | Depth camera uncertainty |
| Link mass | ±10% | Manufacturing tolerance |
| Joint friction | 0.5–2× nominal | Thermal and wear variation |

---

### 5.6 Beyond Domain Randomization

- **Residual policy learning:** Deploy the sim-trained base policy on hardware, collect real trajectories, train a small residual correction network on real data to fix systematic biases such as persistent calibration offset.
- **Real-to-sim adaptation:** Use real hardware trajectories to estimate true actuator dynamics and friction parameters, update the simulation, and retrain. The reBot-DevArm's Isaac Sim integration (planned Q2 2026) is the intended path for this on this platform.

## 6. Publish Artifacts

In [ ]:
artifacts = ['results/plots/training_curves_v1.png']

videos = sorted(Path(VIDEOS_DIR).glob('*.mp4'))
if videos:
    artifacts.append(str(videos[-1].relative_to(REPO)))

nb = save_notebook('challenge3-v1.ipynb')
if nb:
    artifacts.append(nb)

publish_artifacts(artifacts, 'add training curves, rollout video, and notebook')